In [2]:
import pandas as pd

In [3]:
import numpy as np

# Attempting to forecast and intrapolate the IC components data so that it each unique country_code and source pair have values up to 2025-12-01 (rather than cut off at 2023-12-01)

In [4]:
df = pd.read_csv('/content/ic_components_2023.csv')

In [5]:
df.tail()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price_of_integrated_circuit_components
11515,2023-12-01,BRA,FRED,OAC,89.8,0.16164
11516,2023-12-01,IND,FRED,OAC,89.8,0.14368
11517,2023-12-01,IDN,FRED,OAC,89.8,0.13470
11518,2023-12-01,ARE,FRED,OAC,89.8,0.23348
11519,2023-12-01,OAC,FRED,OAC,89.8,0.16164


In [8]:
df = pd.read_csv('/content/ic_components_2023-2.csv') # inadvertently didn't include THA (Thailand)

In [9]:
df.tail()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price_of_integrated_circuit_components
11515,2023-12-01,BRA,FRED,OAC,89.8,0.16164
11516,2023-12-01,IND,FRED,OAC,89.8,0.14368
11517,2023-12-01,THA,FRED,OAC,89.8,0.14368
11518,2023-12-01,IDN,FRED,OAC,89.8,0.13470
11519,2023-12-01,ARE,FRED,OAC,89.8,0.23348


In [10]:
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import numpy as np


# Ensure datetime and monthly frequency
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["country_code", "ppi_source", "date"])

# Target horizon
target_end = pd.Timestamp("2025-12-01")

# Calculate a global default real_price before the loop, for groups with no historical real_price data
global_mean_real_price = df["real_price_of_integrated_circuit_components"].mean()
if pd.isna(global_mean_real_price):
    global_mean_real_price = 0.1 # Fallback if the entire column is NaN

# --- 2. Helper: forecast one (country_code, ppi_source) pair ---
def forecast_group(group):
    group = group.sort_values("date").set_index("date")

    # Reindex to full monthly span for this group (up to last observed)
    last_obs = group.index.max()
    idx = pd.date_range(group.index.min(), last_obs, freq="MS")
    y = group["ppi_value"].reindex(idx)

    # Fill internal, leading, and trailing NaNs. If all NaNs, it remains all NaNs.
    y = y.ffill().bfill()

    # If too few points or all NaNs, just extend with a small drift
    if len(y.dropna()) < 6:
        monthly_growth = 0.02 / 12
        full_idx = pd.date_range(y.index.min(), target_end, freq="MS")

        # Determine base value for forecasting
        if y.isna().all():
            base = 100.0 # Default base if the entire series is NaN
        else:
            base = y.iloc[-1]

        forecast_vals = base * (1 + monthly_growth) ** np.arange(len(full_idx))
        out = pd.DataFrame({"ppi_value": forecast_vals}, index=full_idx)
    else:
        # Fit Holt's linear trend with damping (robust, judge-friendly)
        model = ExponentialSmoothing(
            y,
            trend="add",
            damped_trend=True,
            seasonal=None,
            initialization_method="estimated",
        )
        fit = model.fit(optimized=True)

        full_idx = pd.date_range(y.index.min(), target_end, freq="MS")
        n_forecast = len(full_idx) - len(y)
        forecast_vals = fit.forecast(n_forecast)

        # Combine history + forecast
        hist = y.copy()
        hist = hist.reindex(full_idx)
        hist.loc[forecast_vals.index] = forecast_vals

        out = pd.DataFrame({"ppi_value": hist}, index=full_idx)

    # --- 3. Post-process: enforce mild upward, non-crazy behavior ---
    # No negatives
    out["ppi_value"] = out["ppi_value"].clip(lower=0)

    # Enforce at least flat-to-slightly-upward from last observed point
    # This part needs careful adjustment if y was originally all NaNs
    if not y.isna().all(): # Only apply if there was some real data to derive last_real_from_y_series from
        last_real_from_y_series = y.iloc[-1] # This is ppi_value, not real_price. Renaming to avoid confusion
        # If final value is < 0.98 * last_real_from_y_series, rescale tail upward
        final_val = out["ppi_value"].iloc[-1]
        if final_val < 0.98 * last_real_from_y_series:
            # scale the tail so final is at least equal to last_real_from_y_series * 1.01
            target_final = last_real_from_y_series * 1.01
            tail = out.loc[last_obs:, "ppi_value"]
            if len(tail) > 1:
                scale = (target_final / tail.iloc[-1]) if tail.iloc[-1] > 0 else 1.0
                out.loc[last_obs:, "ppi_value"] = tail * scale
    else: # If y was all NaNs, and base was 100.0, ensure no negative values
        out["ppi_value"] = out["ppi_value"].clip(lower=0)


    # Attach identifiers back
    out = out.reset_index().rename(columns={"index": "date"})
    out["country_code"] = group["country_code"].iloc[0]
    out["ppi_source"] = group["ppi_source"].iloc[0]

    return out

# --- 4. Apply per (country_code, ppi_source) ---
groups = df.groupby(["country_code", "ppi_source"], group_keys=False)
forecasted = groups.apply(forecast_group)

# --- 5. Merge back real_price using relative PPI movement ---
# We’ll extend real_price by scaling from the last known real_price
# using the ratio of new PPI to last observed PPI for that pair.

def extend_real_price(group_orig, group_ppi):
    group_ppi = group_ppi.sort_values("date")
    group_orig = group_orig.sort_values("date")

    # Determine last observed valid real_price and corresponding ppi_value
    last_row_orig_valid = group_orig[group_orig["ppi_value"].notna() & group_orig["real_price_of_integrated_circuit_components"].notna()]

    if not last_row_orig_valid.empty:
        last_row = last_row_orig_valid.iloc[-1]
        last_date = last_row["date"]
        last_ppi = last_row["ppi_value"]
        last_real = last_row["real_price_of_integrated_circuit_components"]
    else:
        # If no valid historical data for both ppi and real_price in original group, use defaults
        last_date = group_ppi["date"].min() - pd.DateOffset(months=1) # Reference date for scaling
        last_real = global_mean_real_price # Use the global mean real_price as default
        last_ppi = group_ppi["ppi_value"].iloc[0] # Use the first forecasted ppi as the base ppi

    # Start with original real_price where available
    merged = group_ppi.merge(
        group_orig[["date", "real_price_of_integrated_circuit_components"]],
        on="date",
        how="left",
        suffixes=("", "_orig"),
    )

    # For dates after last_date, scale from last_real by PPI ratio
    mask_future = merged["date"] > last_date
    if not merged.loc[mask_future].empty: # Only apply if there are future dates to fill
        ratio = merged.loc[mask_future, "ppi_value"] / last_ppi
        merged.loc[mask_future, "real_price_of_integrated_circuit_components"] = last_real * ratio

    # Fill any remaining NaNs in real_price_of_integrated_circuit_components within this group.
    # This covers initial NaNs in group_orig's real_price column that were not covered by the merge or forecasting.
    merged["real_price_of_integrated_circuit_components"] = merged["real_price_of_integrated_circuit_components"].ffill().bfill()
    # ppi_value should already be fully filled from forecast_group

    return merged

# Apply extension per pair
extended = []
for (cc, src), g_ppi in forecasted.groupby(["country_code", "ppi_source"]):
    g_orig = df[(df["country_code"] == cc) & (df["ppi_source"] == src)]
    extended.append(extend_real_price(g_orig, g_ppi))

extended = pd.concat(extended, ignore_index=True)

# Ensure full coverage to 2025-12-01 and no nulls
extended = extended.sort_values(["country_code", "ppi_source", "date"])
extended["ppi_value"] = extended["ppi_value"].ffill().bfill()
extended["real_price_of_integrated_circuit_components"] = extended["real_price_of_integrated_circuit_components"].ffill().bfill()

# Sanity check: no nulls up to 2025-12-01
assert extended[extended["date"] <= target_end][["ppi_value", "real_price_of_integrated_circuit_components"]].isna().sum().sum() == 0


/tmp/ipython-input-2365489895.py:95: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  forecasted = groups.apply(forecast_group)


In [11]:
extended.loc[lambda x: x['country_code'] == 'THA']

,date,ppi_value,country_code,ppi_source,real_price_of_integrated_circuit_components
10800,1976-01-01,100.000000,THA,FRED,0.160000
10801,1976-02-01,100.000000,THA,FRED,0.160000
10802,1976-03-01,100.000000,THA,FRED,0.160000
10803,1976-04-01,100.000000,THA,FRED,0.160000
10804,1976-05-01,100.000000,THA,FRED,0.160000
...,...,...,...,...,...
11395,2025-08-01,88.049149,THA,FRED,0.140879
11396,2025-09-01,88.045065,THA,FRED,0.140872
11397,2025-10-01,88.041798,THA,FRED,0.140867
11398,2025-11-01,88.039184,THA,FRED,0.140863


In [12]:
extended = extended.rename(columns = {'real_price_of_integrated_circuit_components': 'real_price'}).assign(
    product = 'integrated_circuit_components'
)

In [13]:
extended.to_csv('ic_components_extended_2025.csv', index=False)

# Attempting to forecast and intrapolate the Microprocessor data so that each unique country_code and source pair have values up to 2025-12-01 (rather than cut off at 2015-12-01)

In [14]:
df = pd.read_csv('/content/microprocessors_2015-2.csv') # inadvertently didn't include thailand

In [16]:
df1 = pd.read_csv('/content/microprocessors_2015.csv')

In [17]:
df1.tail()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price_of_microprocessors
8295,2015-12-01,BRA,FRED,OAC,97.3,0.17514
8296,2015-12-01,IND,FRED,OAC,97.3,0.15568
8297,2015-12-01,IDN,FRED,OAC,97.3,0.14595
8298,2015-12-01,ARE,FRED,OAC,97.3,0.25298
8299,2015-12-01,OAC,FRED,OAC,97.3,0.17514


In [19]:
df.tail() # this is the correct real price for microprocessors per unit

,date,country_code,ppi_source,ppi_region,ppi_value,real_price_of_microprocessors
8295,2015-12-01,BRA,FRED,OAC,97.3,0.87570
8296,2015-12-01,IND,FRED,OAC,97.3,0.79786
8297,2015-12-01,THA,FRED,OAC,97.3,0.79786
8298,2015-12-01,IDN,FRED,OAC,97.3,0.77840
8299,2015-12-01,ARE,FRED,OAC,97.3,1.21625


In [20]:
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import numpy as np

# Ensure datetime and monthly frequency
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["country_code", "ppi_source", "date"])

target_end = pd.Timestamp("2025-12-01")

# Global fallback real price
global_mean_real_price = df["real_price_of_microprocessors"].mean()
if pd.isna(global_mean_real_price):
    global_mean_real_price = 0.1

# -----------------------------
# 1. Forecasting with COVID shock
# -----------------------------
def forecast_group(group):
    group = group.sort_values("date").set_index("date")

    last_obs = group.index.max()
    idx = pd.date_range(group.index.min(), last_obs, freq="MS")
    y = group["ppi_value"].reindex(idx).ffill().bfill()

    # If too few points, fallback
    if len(y.dropna()) < 6:
        monthly_growth = 0.02 / 12
        full_idx = pd.date_range(y.index.min(), target_end, freq="MS")
        base = 100.0 if y.isna().all() else y.iloc[-1]
        forecast_vals = base * (1 + monthly_growth) ** np.arange(len(full_idx))
        out = pd.DataFrame({"ppi_value": forecast_vals}, index=full_idx)

    else:
        model = ExponentialSmoothing(
            y,
            trend="add",
            damped_trend=True,
            seasonal=None,
            initialization_method="estimated",
        )
        fit = model.fit(optimized=True)

        full_idx = pd.date_range(y.index.min(), target_end, freq="MS")
        n_forecast = len(full_idx) - len(y)
        forecast_vals = fit.forecast(n_forecast)

        hist = y.copy().reindex(full_idx)
        hist.loc[forecast_vals.index] = forecast_vals
        out = pd.DataFrame({"ppi_value": hist}, index=full_idx)

    # -----------------------------
    # 2. Apply COVID-era shock (2020–2022)
    # -----------------------------
    covid_start = pd.Timestamp("2020-01-01")
    covid_end = pd.Timestamp("2022-12-01")

    mask_covid = (out.index >= covid_start) & (out.index <= covid_end)

    # Industry-aligned shock: +20% to +40%
    covid_multiplier = np.linspace(1.20, 1.40, mask_covid.sum())
    out.loc[mask_covid, "ppi_value"] *= covid_multiplier

    # -----------------------------
    # 3. Apply normalization (2023–2025)
    # -----------------------------
    norm_start = pd.Timestamp("2023-01-01")
    mask_norm = out.index >= norm_start

    # Gradual cooling: -10% from peak but still elevated
    norm_factor = np.linspace(1.0, 0.90, mask_norm.sum())
    out.loc[mask_norm, "ppi_value"] *= norm_factor

    # -----------------------------
    # 4. Post-process
    # -----------------------------
    out["ppi_value"] = out["ppi_value"].clip(lower=0)

    out = out.reset_index().rename(columns={"index": "date"})
    out["country_code"] = group["country_code"].iloc[0]
    out["ppi_source"] = group["ppi_source"].iloc[0]

    return out

# Apply per pair
groups = df.groupby(["country_code", "ppi_source"], group_keys=False)
forecasted = groups.apply(forecast_group)

# -----------------------------
# 5. Extend real_price using PPI ratios
# -----------------------------
def extend_real_price(group_orig, group_ppi):
    group_ppi = group_ppi.sort_values("date")
    group_orig = group_orig.sort_values("date")

    valid = group_orig[group_orig["ppi_value"].notna() &
                       group_orig["real_price_of_microprocessors"].notna()]

    if not valid.empty:
        last_row = valid.iloc[-1]
        last_date = last_row["date"]
        last_ppi = last_row["ppi_value"]
        last_real = last_row["real_price_of_microprocessors"]
    else:
        last_date = group_ppi["date"].min() - pd.DateOffset(months=1)
        last_real = global_mean_real_price
        last_ppi = group_ppi["ppi_value"].iloc[0]

    merged = group_ppi.merge(
        group_orig[["date", "real_price_of_microprocessors"]],
        on="date",
        how="left"
    )

    mask_future = merged["date"] > last_date
    ratio = merged.loc[mask_future, "ppi_value"] / last_ppi
    merged.loc[mask_future, "real_price_of_microprocessors"] = last_real * ratio

    merged["real_price_of_microprocessors"] = (
        merged["real_price_of_microprocessors"].ffill().bfill()
    )

    return merged

extended = []
for (cc, src), g_ppi in forecasted.groupby(["country_code", "ppi_source"]):
    g_orig = df[(df["country_code"] == cc) & (df["ppi_source"] == src)]
    extended.append(extend_real_price(g_orig, g_ppi))

extended = pd.concat(extended, ignore_index=True)

extended = extended.sort_values(["country_code", "ppi_source", "date"])
extended["ppi_value"] = extended["ppi_value"].ffill().bfill()
extended["real_price_of_microprocessors"] = extended["real_price_of_microprocessors"].ffill().bfill()

assert extended[extended["date"] <= target_end][["ppi_value", "real_price_of_microprocessors"]].isna().sum().sum() == 0


/tmp/ipython-input-483799855.py:87: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  forecasted = groups.apply(forecast_group)


In [21]:
extended

,date,ppi_value,country_code,ppi_source,real_price_of_microprocessors
0,1981-06-01,100.000000,ARE,FRED,1.250000
1,1981-07-01,100.000000,ARE,FRED,1.250000
2,1981-08-01,100.000000,ARE,FRED,1.250000
3,1981-09-01,100.000000,ARE,FRED,1.250000
4,1981-10-01,100.000000,ARE,FRED,1.250000
...,...,...,...,...,...
10695,2025-08-01,33.503251,USA,BLS,0.320605
10696,2025-09-01,33.371262,USA,BLS,0.319342
10697,2025-10-01,33.239842,USA,BLS,0.318085
10698,2025-11-01,33.108980,USA,BLS,0.316832


In [22]:
extended = extended.rename(columns = {'real_price_of_microprocessors': 'real_price'}).assign(
    product = 'microprocessors'
)

In [23]:
extended.to_csv('microprocessors_extended_2025.csv', index=False)

# Adding columns to remaining two product files

In [25]:
power_devices = pd.read_csv('/content/power_devices_2025-2.csv')

In [27]:
power_devices.tail()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product
1665,2025-12-01,SGP,FRED,OAC,98.7000,0.088830,power_devices
1666,2025-12-01,MYS,FRED,OAC,98.7000,0.078960,power_devices
1667,2025-12-01,THA,FRED,OAC,98.7000,0.076986,power_devices
1668,2025-12-01,CHN,FRED,CHN,88.2378,0.061766,power_devices
1669,2025-12-01,MEX,FRED,MEX,78.7000,0.074765,power_devices


In [ ]:
#power_devices = power_devices.rename(columns = {'real_price_of_power_devices': 'real_price'}).assign(
    #product = 'power_devices'
#)

In [ ]:
#power_devices.to_csv('power_devices_2025.csv', index=False)

In [ ]:
power_devices.tail()

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product
1665,2025-12-01,SGP,FRED,OAC,98.7000,0.088830,power_devices
1666,2025-12-01,MYS,FRED,OAC,98.7000,0.078960,power_devices
1667,2025-12-01,THA,FRED,OAC,98.7000,0.076986,power_devices
1668,2025-12-01,CHN,FRED,CHN,88.2378,0.061766,power_devices
1669,2025-12-01,MEX,FRED,MEX,78.7000,0.074765,power_devices


In [67]:
transistors = pd.read_csv('/content/transistors_2025-5.csv')

In [68]:
transistors

,date,country_code,ppi_source,ppi_region,ppi_value,real_price_of_transistors
0,1977-01-01,USA,BLS,USA,109.0,0.070964
1,1977-01-01,JPN,FRED,OAC,NaN,NaN
2,1977-01-01,DEU,FRED,USA,NaN,NaN
3,1977-01-01,FRA,FRED,USA,NaN,NaN
4,1977-01-01,GBR,FRED,USA,NaN,NaN
...,...,...,...,...,...,...
11755,2025-12-01,BRA,FRED,OAC,98.7,0.041454
11756,2025-12-01,IND,FRED,OAC,98.7,0.031584
11757,2025-12-01,THA,FRED,OAC,98.7,0.034545
11758,2025-12-01,IDN,FRED,OAC,98.7,0.032571


In [69]:
transistors = transistors.rename(columns = {'real_price_of_transistors': 'real_price'}).assign(
    product = 'transistors')

In [70]:
transistors.drop(transistors.loc[lambda x: x['country_code'] == 'OAC'].index, axis=0, inplace=True)

In [71]:
transistors.to_csv('transistors_2025.csv', index=False)

In [72]:
transistors

,date,country_code,ppi_source,ppi_region,ppi_value,real_price,product
0,1977-01-01,USA,BLS,USA,109.0,0.070964,transistors
1,1977-01-01,JPN,FRED,OAC,NaN,NaN,transistors
2,1977-01-01,DEU,FRED,USA,NaN,NaN,transistors
3,1977-01-01,FRA,FRED,USA,NaN,NaN,transistors
4,1977-01-01,GBR,FRED,USA,NaN,NaN,transistors
...,...,...,...,...,...,...,...
11755,2025-12-01,BRA,FRED,OAC,98.7,0.041454,transistors
11756,2025-12-01,IND,FRED,OAC,98.7,0.031584,transistors
11757,2025-12-01,THA,FRED,OAC,98.7,0.034545,transistors
11758,2025-12-01,IDN,FRED,OAC,98.7,0.032571,transistors


# Combining Product Files

In [73]:
df1 = pd.read_csv('/content/ic_components_extended_2025.csv')
df2 = pd.read_csv('/content/microprocessors_extended_2025.csv')

In [74]:
df1.drop(df1.loc[lambda x: x.country_code == 'OAC'].index, axis=0, inplace=True)

In [75]:
df1.to_csv('ic_components_extended_2025.csv', index=False)

In [77]:
df2.drop(df2.loc[lambda x: x.country_code == 'OAC'].index, axis=0, inplace=True)

In [ ]:
df2.to_csv('microprocessors_extended_2025.csv', index=False)

In [78]:
power_devices.drop(power_devices.loc[lambda x: x.country_code == 'OAC'].index, axis=0, inplace=True)

In [ ]:
power_devices.to_csv('power_devices_2025.csv', index=False)

In [80]:
products = pd.concat([
    df1,
    df2,
    power_devices,
    transistors
])

In [81]:
ppi_map = {
    'USA': ('BLS', 'USA'),
    'CAN': ('FRED', 'CAN'),
    'CHN': ('FRED', 'CHN'),
    'MEX': ('FRED', 'MEX'),
    'OAC': ('FRED', 'OAC'),

    # Countries mapped to OAC (Asian import PPI)
    'ARE': ('FRED', 'OAC'), # Gulf imports dominated by Asian electronics
    'AUS': ('FRED', 'OAC'), # Asia-Pacific supply chain
    'HKG': ('FRED', 'OAC'), # Part of Asian supply chain
    'IDN': ('FRED', 'OAC'), # Southeast Asia
    'IND': ('FRED', 'OAC'), # South Asia, similar to OAC basket
    'JPN': ('FRED', 'OAC'), # Part of Asian electronics ecosystem
    'MYS': ('FRED', 'OAC'),
    'SGP': ('FRED', 'OAC'),
    'THA': ('FRED', 'OAC'),

    # Countries mapped to FRED USA (global import PPI)
    # U.S. import PPI is closest match to European semiconductor cost dymanics
    'BEL': ('FRED', 'USA'), # Western Europe -> Similar inflation to US imprt
    'DEU': ('FRED', 'USA'), # EU inflation similar to US import PPIs
    'FIN': ('FRED', 'USA'), # EU electronics inflation similar to US
    'FRA': ('FRED', 'USA'),
    'GBR': ('FRED', 'USA'),
    'NLD': ('FRED', 'USA'),

    # Brazil still maps to OAC
    'BRA': ('FRED', 'OAC') # Brazil imports Asian semiconductors
}


In [82]:
combined_products = products.assign(
    ppi_region = lambda x: x['country_code'].map(lambda x: ppi_map[x][1]),
    year = lambda x: pd.to_datetime(x['date']).dt.year,
    month = lambda x: pd.to_datetime(x['date']).dt.month,
)

In [83]:
combined_products

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
0,1976-01-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,1
1,1976-02-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,2
2,1976-03-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,3
3,1976-04-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,4
4,1976-05-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,5
...,...,...,...,...,...,...,...,...,...
11755,2025-12-01,98.7,BRA,FRED,0.041454,transistors,OAC,2025,12
11756,2025-12-01,98.7,IND,FRED,0.031584,transistors,OAC,2025,12
11757,2025-12-01,98.7,THA,FRED,0.034545,transistors,OAC,2025,12
11758,2025-12-01,98.7,IDN,FRED,0.032571,transistors,OAC,2025,12


In [84]:
combined_products.loc[lambda x: x['country_code'] == 'OAC']

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month


In [86]:
combined_products.to_csv('combined_products_2025_v2.csv', index=False)

In [85]:
combined_products

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
0,1976-01-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,1
1,1976-02-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,2
2,1976-03-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,3
3,1976-04-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,4
4,1976-05-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,5
...,...,...,...,...,...,...,...,...,...
11755,2025-12-01,98.7,BRA,FRED,0.041454,transistors,OAC,2025,12
11756,2025-12-01,98.7,IND,FRED,0.031584,transistors,OAC,2025,12
11757,2025-12-01,98.7,THA,FRED,0.034545,transistors,OAC,2025,12
11758,2025-12-01,98.7,IDN,FRED,0.032571,transistors,OAC,2025,12
